In [2]:
words= open("../data/shakespeare_100k.txt", "r").read().splitlines()

In [3]:
import torch

N = torch.zeros((52,52),dtype=torch.int32) 

lookup=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(lookup)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}

# initialise N matrix

for w in words:
    chs=["."] + list(w) + ["."] 
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        N[ix1,ix2]+=1


P=(N+1).float() # did (N+1) it does scalar addition of 1 to every elemnt of co-occurrence matrix so no prabability is exactly zero it is called model smoothing
P=P/P.sum(1,keepdim=True) # converting numbers in the matrix to probabilities

g= torch.Generator().manual_seed(214748367)

for i in range(20):
    out=[]
    ix=0
    while True:
        p=P[ix].float() # P is the matrix containing the probability distribution
        ix=torch.multinomial(p,num_samples=1,replacement=True,generator=g).item()

        out.append(itos[ix])

        if ix==0:
            break
    print(''.join(out))



sar.
ONTRD.
thooysingoroouigJONDUKEngrokelofathxthu.
ves.
k.
whe.
t.
they.
Noo.
foulde.
wimmenoot.
tre.
Rof.
fofofourt.
t.
bribris.
theath.
t.
IONEr.
con.


# Loss function bigram model 

In [4]:
log_likelihood=0.0
n=0.0
for w in words:
    chs=["."] + list(w) + ["."] 
    for ch1, ch2 in zip(chs,chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        prob=P[ix1, ix2]
        logprob=torch.log(prob) # log probabilities
        log_likelihood += logprob
        n+=1
        print(f"{ch1}{ch2}:  {prob:.4f}  {logprob:.4f}")

print(f'{log_likelihood=}') # our goal will be to minimize the negative log_likelihood
nll=-log_likelihood
avg_nll=(f"{nll/n}")# this is the avg negative log likelihood and we can minimize this instead
print(f'{avg_nll=}')

.T:  0.0243  -3.7166
TH:  0.0300  -3.5081
HE:  0.1449  -1.9314
E.:  0.1267  -2.0657
.S:  0.0141  -4.2614
SO:  0.0321  -3.4382
ON:  0.1225  -2.0994
NN:  0.0024  -6.0434
NE:  0.0700  -2.6590
ET:  0.0045  -5.3979
TS:  0.0011  -6.7946
S.:  0.2563  -1.3612
.A:  0.0248  -3.6978
AL:  0.0469  -3.0597
LL:  0.0894  -2.4151
L.:  0.0096  -4.6422
.I:  0.0334  -3.3982
IS:  0.0012  -6.7176
S.:  0.2563  -1.3612
.W:  0.0144  -4.2431
WE:  0.0036  -5.6384
EL:  0.0724  -2.6253
LL:  0.0894  -2.4151
L.:  0.0096  -4.6422
.T:  0.0243  -3.7166
TH:  0.0300  -3.5081
HA:  0.0663  -2.7135
AT:  0.0445  -3.1113
T.:  0.0641  -2.7472
.E:  0.0089  -4.7233
EN:  0.1491  -1.9033
ND:  0.1642  -1.8069
DS:  0.0047  -5.3673
S.:  0.2563  -1.3612
.W:  0.0144  -4.2431
WE:  0.0036  -5.6384
EL:  0.0724  -2.6253
LL:  0.0894  -2.4151
L.:  0.0096  -4.6422
.T:  0.0243  -3.7166
TH:  0.0300  -3.5081
HE:  0.1449  -1.9314
E.:  0.1267  -2.0657
.T:  0.0243  -3.7166
TR:  0.0927  -2.3788
RA:  0.1705  -1.7691
AG:  0.0092  -4.6861
GE:  0.1512  

Till now we used co-occurence matrix to predict the next char but we are going to implement the bigram in Neural Network settings with paramaters so that we can minize the loss and make better model for predicting next char 

In [5]:
# creating the training set of bigram(x,y)
# this will create a dataset with every bigram ex for a bigram 'wi' x=w and y=i similarly for all possible pair
xs, ys = [], []

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        print(ch1,ch2)
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)


. T
T H
H E
E .


In [6]:
xs

tensor([ 0, 20,  8,  5])

In [7]:
ys

tensor([20,  8,  5,  0])

In [8]:
# now our xs contains numerical (itegers) value mapped to a numerical value in ys but integer itself 
# can not be fed into the neural network we nee some encoding to represent that numerical value with
# context of it being a word in our vocabulary of 51 words we can different encodings such as one-hot,
# learned embeddings,etc

#one-hot embedding

import torch.nn.functional as F 
X=F.one_hot(xs, num_classes=51).float()

X.shape


torch.Size([4, 51])

In [9]:
# this is learned embedding in this embedding each character is represented by 8dim vector
# for example word s = [0.23,1.3,0.97,2.4......] unlike one hot encoding where evry word is represented using 51dim one-hot vector
# this embedding is better but we would not use this in this notebook
# import torch
# import torch.nn as nn

# # Vocabulary size and embedding dimensions
# vocab_size = 51
# embedding_dim = 8

# # Create embedding layer
# embedding = nn.Embedding(vocab_size, embedding_dim)

# # Convert input indices into embeddings
# X = embedding(xs)

# print(X)

In [10]:
# initialising the weights

W=torch.randn((51,51)) # this creates 8X20 matrix of weights which will be multiplied by our embedding vector here 20 represents 20 neurons in the input layer
                     # therefore size 8X20 as 51X8 * 8X20 = 51X20 output 
                     # torch.randn gives random numbers from a normal distribution with mean=0


what W contains is 8 rows with 20 columns that there are 8 different weight vectors containing 20 different weights for each neuron 

[
    [w1-1,w1-2,w1-3......w1-20],
    [w2-1,w2-2,w2-3......w2-20],
    .
    .
    .
    ......................w8-20]
]

where w2-3 = w2 of neuron 3
      w4-13 = w4 of neuron 13

In [11]:
(X@W)[1]

tensor([ 1.8296,  1.2376, -1.6496,  1.8891,  1.2804, -1.2475,  0.3982, -0.8447,
         0.3027,  2.3864, -1.8022,  1.1813,  0.9187, -0.1737, -1.1594,  0.3743,
         0.5287, -2.0958,  0.8678, -0.7979, -1.1302, -0.8584, -0.7993, -1.1570,
         1.1774, -0.2153, -1.8609,  0.2202, -0.0718,  1.1727,  0.9129,  0.4809,
         0.0124, -0.7640,  0.4462,  0.7950, -1.3233,  0.8184, -0.8598, -1.3825,
        -0.4005, -0.0887,  1.2767, -0.6027, -0.3623,  0.7273, -2.2234, -0.6619,
         0.3188, -1.0881,  0.6836])

In [12]:
(X@W)[2,13]# this gives fiering rate of 13th neuron considering the 2nd input
# Basically it is the multiplication(dot prod) of 3rd row of the embedding vector and weights vector for the 13th neuron
# @ is used to compute matrix multiplication

tensor(-0.5615)

In [13]:
(X@W).exp()

tensor([[ 0.5370,  0.6428,  0.1646,  1.0669,  0.3920,  0.6749,  0.4873,  0.3294,
          5.3298,  2.6222,  0.2930,  0.3530,  0.8076,  0.3544,  0.7906,  0.9191,
          1.1028,  1.2016,  8.9118,  3.8512,  1.4150,  0.6180,  0.8615,  0.3443,
          1.7408,  2.5761,  2.3186,  0.2771,  2.4973,  0.4779,  1.4132,  1.3848,
          1.8717,  0.4583,  0.6749,  0.6775,  1.6996,  0.7065,  0.2434,  0.6992,
          5.0501,  2.5716,  3.8527,  1.0543,  1.1291,  0.2159,  1.4681,  0.8905,
          1.8775,  0.9159,  2.0411],
        [ 6.2314,  3.4474,  0.1921,  6.6136,  3.5979,  0.2872,  1.4892,  0.4297,
          1.3535, 10.8745,  0.1649,  3.2585,  2.5060,  0.8405,  0.3137,  1.4539,
          1.6967,  0.1230,  2.3817,  0.4503,  0.3230,  0.4239,  0.4497,  0.3144,
          3.2460,  0.8063,  0.1555,  1.2464,  0.9308,  3.2308,  2.4916,  1.6175,
          1.0124,  0.4658,  1.5623,  2.2145,  0.2662,  2.2669,  0.4232,  0.2509,
          0.6700,  0.9151,  3.5846,  0.5473,  0.6961,  2.0696,  0.1082, 

In [14]:
logits = X@W # log counts
counts=logits.exp() # equivalent to N (makes evrything possitive)
probs = counts/counts.sum(1,keepdim=True)

probs

tensor([[0.0072, 0.0086, 0.0022, 0.0143, 0.0052, 0.0090, 0.0065, 0.0044, 0.0712,
         0.0350, 0.0039, 0.0047, 0.0108, 0.0047, 0.0106, 0.0123, 0.0147, 0.0161,
         0.1191, 0.0514, 0.0189, 0.0083, 0.0115, 0.0046, 0.0233, 0.0344, 0.0310,
         0.0037, 0.0334, 0.0064, 0.0189, 0.0185, 0.0250, 0.0061, 0.0090, 0.0091,
         0.0227, 0.0094, 0.0033, 0.0093, 0.0675, 0.0344, 0.0515, 0.0141, 0.0151,
         0.0029, 0.0196, 0.0119, 0.0251, 0.0122, 0.0273],
        [0.0740, 0.0409, 0.0023, 0.0785, 0.0427, 0.0034, 0.0177, 0.0051, 0.0161,
         0.1291, 0.0020, 0.0387, 0.0298, 0.0100, 0.0037, 0.0173, 0.0201, 0.0015,
         0.0283, 0.0053, 0.0038, 0.0050, 0.0053, 0.0037, 0.0385, 0.0096, 0.0018,
         0.0148, 0.0111, 0.0384, 0.0296, 0.0192, 0.0120, 0.0055, 0.0186, 0.0263,
         0.0032, 0.0269, 0.0050, 0.0030, 0.0080, 0.0109, 0.0426, 0.0065, 0.0083,
         0.0246, 0.0013, 0.0061, 0.0163, 0.0040, 0.0235],
        [0.0106, 0.0493, 0.0176, 0.0457, 0.0201, 0.1201, 0.0016, 0.0268, 0

In [15]:
# SUMMARY------------------------------------------>>>>>>>>>>>>>>>>>>>

In [16]:
xs

tensor([ 0, 20,  8,  5])

In [17]:
ys

tensor([20,  8,  5,  0])

In [18]:
#randomly initialize 20 neurons with only ws no# equivalent to N biases
W=torch.randn((51,51))

In [19]:
logits = X@W # log counts
counts=logits.exp() # equivalent to N
probs = counts/counts.sum(1,keepdim=True) # probabilities for next character

# BTW: the last 2 lines together are softmax

In [20]:
probs.shape

torch.Size([4, 51])

In [21]:
# nlls = torch.zeros(4)
# for i in range(4):
#   # i-th bigram:
#   x = xs[i].item() # input character index
#   y = ys[i].item() # label character index
#   print('--------')
#   print(f'bigram example {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
#   print('input to the neural net:', x)
#   print('output probabilities from the neural net:', probs[i])
#   print('label (actual next character):', y)
#   p = probs[i, y]
#   print('probability assigned by the net to the the correct character:', p.item())
#   logp = torch.log(p)
#   print('log likelihood:', logp.item())
#   nll = -logp
#   print('negative log likelihood:', nll.item())
#   nlls[i] = nll

# print('=========')
# print('average negative log likelihood, i.e. loss =', nlls.mean().item())

In [22]:
#------------!!!!OPTIMIZATION!!------------- 

In [23]:
xs

tensor([ 0, 20,  8,  5])

In [24]:
ys

tensor([20,  8,  5,  0])

In [25]:
W = torch.randn((51, 51), requires_grad=True)

In [2341]:
# forward pass
X= F.one_hot(xs, num_classes=51).float() # input to the network: one-hot encoding
logits = X @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
loss = -probs[torch.arange(4), ys].log().mean()


In [2342]:
# backward pass
W.grad=None # set gradient to zero
loss.backward()

In [2343]:
#parameters update
W.data+= -0.1*W.grad